# Vectorizar el corpus del BCV con embeddings de OpenAI

Genera los **vectores del corpus jurídico** para activar la búsqueda semántica en
el portal desplegado en Vercel, y **mide su calidad** con las mismas consultas que
se usaron para los otros modelos.

**¿Por qué desde Colab?** OpenAI bloquea por país (`unsupported_country_region_territory`).
Colab sale a internet desde regiones admitidas. El portal, además, ejecuta la
consulta desde los servidores de Vercel, no desde tu conexión.

| | |
|---|---|
| **Produce** | `web/api/_data/vectors.f32` y `embeddings_meta.json` |
| **Coste** | ~2 000 fragmentos ≈ 1 millón de tokens ≈ **unos pocos céntimos** |
| **Frecuencia** | una sola vez (y de nuevo solo si cambia el corpus) |

> No hace falta instalar nada: los scripts usan solo la biblioteca estándar de
> Python y numpy (ya viene en Colab).

## 1. Traer el repositorio

In [ ]:
REPO = "https://github.com/uptaragua-oficial/bcv-conocimiento-vectorial.git"

!git clone --depth 1 {REPO} 2>/dev/null || (cd bcv-conocimiento-vectorial && git pull)
%cd bcv-conocimiento-vectorial

!ls web/api/_data/
print()
!python -c "import json; d=json.load(open('web/api/_data/corpus.json')); print('fragmentos en el corpus:', d['n'])"

## 2. Introducir la clave de OpenAI

`getpass` evita que la clave quede visible en la salida del cuaderno.
Créala en <https://platform.openai.com/api-keys>.

Modelos sugeridos:

* `text-embedding-3-small` → 1 536 dimensiones, ~8 MB de vectores (económico)
* `text-embedding-3-large` → 3 072 dimensiones, ~26 MB (más preciso)

In [ ]:
import os
from getpass import getpass

os.environ["EMBEDDINGS_API_KEY"] = getpass("Clave de OpenAI (sk-...): ")
os.environ["EMBEDDINGS_MODEL"] = "text-embedding-3-small"
os.environ["EMBEDDINGS_BASE_URL"] = "https://api.openai.com/v1"

print("Modelo:", os.environ["EMBEDDINGS_MODEL"])
print("Clave cargada:", bool(os.environ["EMBEDDINGS_API_KEY"]))

## 3. Verificar la clave antes de nada

Una sola llamada de prueba. Así sabes de inmediato si el país está bloqueado o
la clave falla, **en lugar de descubrirlo al final**.

In [ ]:
import json, urllib.error, urllib.request

url = f"{os.environ['EMBEDDINGS_BASE_URL'].rstrip('/')}/embeddings"
cuerpo = json.dumps({"model": os.environ["EMBEDDINGS_MODEL"], "input": ["prueba"]}).encode()
req = urllib.request.Request(
    url, data=cuerpo, method="POST",
    headers={"Authorization": f"Bearer {os.environ['EMBEDDINGS_API_KEY']}",
             "Content-Type": "application/json"},
)
try:
    with urllib.request.urlopen(req, timeout=60) as r:
        datos = json.loads(r.read())
    print(f"Clave válida · {len(datos['data'][0]['embedding'])} dimensiones · modelo {datos.get('model')}")
    listo = True
except urllib.error.HTTPError as e:
    print(f"FALLO HTTP {e.code}:")
    print(e.read().decode('utf-8', 'replace')[:500])
    print("\nSi dice 'unsupported_country_region_territory', esta sesión de Colab")
    print("sale por una región no admitida: reinicia el entorno de ejecución y reintenta.")
    listo = False
assert listo, "Corrige la clave o la región antes de continuar."

## 4. Vectorizar el corpus

Verás el avance por lotes. Tarda uno o dos minutos.

In [ ]:
!python -m scripts.export_openai_embeddings

## 5. Medir la calidad y compararla

Ejecuta el comparador con **el mismo conjunto de 40 consultas** que se usó para
los otros modelos, así que la tabla es directamente comparable:

| Referencia medida antes | recall@5 | nDCG@5 |
|---|---:|---:|
| BM25 (léxico) | 0.900 | 0.821 |
| BGE-M3 híbrida | 0.975 | **0.881** |
| e5-large híbrida | 0.925 | 0.852 |

**A batir: 0.881 de nDCG@5.**

In [ ]:
!python -m scripts.comparar_recuperacion \
  --vectores web/api/_data/vectors.f32 \
  --meta     web/api/_data/embeddings_meta.json \
  --nombre   openai-3-small \
  --k 5 --consultas 40 --api

## 6. Descargar los archivos generados

Cópialos a tu repositorio local, en `web/api/_data/`, y haz commit.

En Vercel → *Settings → Environment Variables*:

| Variable | Valor |
|---|---|
| `EMBEDDINGS_API_KEY` | tu clave de OpenAI |
| `EMBEDDINGS_MODEL` | el mismo modelo que usaste aquí |

Redespliega y comprueba que `/api/health` responde `"recuperacion": "hibrida"`.

> ⚠️ Los vectores y la meta deben ser **del mismo modelo**. Si no coinciden con lo
> configurado en Vercel, el portal vuelve a BM25 automáticamente (con un aviso),
> en lugar de devolver resultados incorrectos.

In [ ]:
from google.colab import files

!ls -la web/api/_data/
files.download("web/api/_data/vectors.f32")
files.download("web/api/_data/embeddings_meta.json")